In [2]:
from cobra.io import read_sbml_model
import numpy as np
import pandas as pd

model = read_sbml_model("../models/iJR904.xml.gz")

In [ ]:
carbon_sources = {
    "Glucose": "EX_glc__D_e",
    "Maltose": "EX_malt_e",
    "Galactose": "EX_gal_e",
    "Glycerol": "EX_glyc_e",
    "Lactate": "EX_lac__L_e",
    "Acetate": "EX_ac_e"
}


mean_a_by_carbon_source = {
    "Glucose": 0.0031,
    "Glycerol": 0.0053,
    "Maltose": 0.0040,
    "Galactose": 0.0040,
    "Lactate": 0.0040,
    "Acetate": 0.0040
}


# Base medium
base_medium = model.medium.copy()

for rxn_id in carbon_sources.values():
    base_medium.pop(rxn_id, None)

base_medium["EX_o2_e"] = 999999.0


# Reactions subject to molecular crowding
crowding_reactions = [
    rxn for rxn in model.reactions
    if rxn not in model.exchanges
    and "BIOMASS" not in rxn.id.upper()
]


# Molecular crowding constraint
crowding_constraint = model.problem.Constraint(
    0,
    ub=1.0,
    name="molecular_crowding"
)

model.add_cons_vars(crowding_constraint)
model.solver.update()


# Simulation parameters
n_runs = 1000
beta = 3

rng = np.random.default_rng(seed=42)


results_fbawmc_specific = []


for run in range(n_runs):

    for name, exchange_id in carbon_sources.items():

        # Get carbon-source-specific average crowding coefficient
        mean_a = mean_a_by_carbon_source[name]

        a_values = rng.gamma(
            shape=beta,
            scale=mean_a / beta,
            size=len(crowding_reactions)
        )

        # Assign coefficients to molecular crowding constraint
        coefficients = {}

        for rxn, a_i in zip(crowding_reactions, a_values):

            coefficients[rxn.forward_variable] = a_i
            coefficients[rxn.reverse_variable] = a_i

        crowding_constraint.set_linear_coefficients(coefficients)

        # Define medium for current carbon source
        medium = base_medium.copy()
        uptake_bounds = {
            "Glucose": 10,
            "Galactose": 10,
            "Maltose": 5,
            "Glycerol": 20,
            "Lactate": 20,
            "Acetate": 30
        }

        medium[exchange_id] = uptake_bounds[name]
        model.medium = medium

        solution = model.optimize()

        if solution.status == "optimal":

            growth = solution.objective_value
            uptake = abs(solution.fluxes[exchange_id])

        else:

            growth = np.nan
            uptake = np.nan

        results_fbawmc_specific.append({
            "run": run + 1,
            "carbon_source": name,
            "mean_a": mean_a,
            "growth_rate": growth,
            "uptake_rate": uptake
        })


df_fbawmc_specific = pd.DataFrame(
    results_fbawmc_specific
)

df_fbawmc_specific.head(10)

In [ ]:
summary_fbawmc_specific = (
    df_fbawmc_specific
    .groupby("carbon_source")
    .agg(
        mean_growth=("growth_rate", "mean"),
        std_growth=("growth_rate", "std"),
        mean_uptake=("uptake_rate", "mean"),
        mean_a=("mean_a", "first")
    )
)

summary_fbawmc_specific
summary_fbawmc_specific.to_csv("fbawmc_specific_a_summary.csv")